# Foundation Models for Time Series: A Comprehensive Comparison

This notebook provides a hands-on comparison of **Time Series Foundation Models (TSFMs)** for zero-shot forecasting.

**Models Covered:**
- **TimesFM 2.5** (Google): Decoder-only transformer, 200M parameters *(native quantile forecasting via quantile head)*
- **Chronos 2** (Amazon): T5-based encoder-decoder, 120M parameters *(sampling-based probabilistic forecasts)*
- **Exponential Smoothing** (Traditional baseline) *(probabilistic forecasts)*

**Datasets:**
1. **Air Passengers**: Structured seasonality (monthly, 1949-1960)
2. **Energy Load**: Complex hourly patterns with multi-scale seasonality
3. **Taylor Electricity Demand**: Irregular spikes and stochastic behavior

**What You'll Learn:**
- Zero-shot forecasting without training
- Probabilistic forecasting with confidence intervals (all three models)
- Performance comparison across different data patterns
- When to use each model

**Note:** All three models provide probabilistic forecasts with uncertainty quantification. TimesFM 2.5 uses a native quantile head (10 quantiles: 0.0-0.9), while Chronos 2 uses sampling-based forecasting (21 quantiles).

For conceptual background and architecture details, see the [Foundation Models User Guide](../docs/userguide/foundation_models.md).

## 1. Installation

Install Darts with foundation model support:

In [ ]:
# Uncomment to install:
# !pip install "darts[timesfm,chronos]"
# or with uv:
# !uv pip install "darts[timesfm,chronos]"

## 2. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from darts import TimeSeries
from darts.datasets import AirPassengersDataset, EnergyDataset
from darts.models import TimesFMModel, ChronosModel, ExponentialSmoothing
from darts.metrics import mape

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Configuration
SPLIT_RATIO = 0.8
NUM_SAMPLES = 100

# Define consistent color palette for all plots (using matplotlib default colors)
TIMESFM_COLOR = 'C3'  # Red (distinct from ground truth black)
CHRONOS_COLOR = 'C1'   # Orange
EXP_COLOR = 'C2'       # Green

### Load and Visualize

In [ ]:
# Load dataset
air_series = AirPassengersDataset().load()
air_train, air_val = air_series.split_before(SPLIT_RATIO)

print(f"Total length: {len(air_series)}")
print(f"Training: {len(air_train)} points")
print(f"Validation: {len(air_val)} points")

# Visualize with train/test split
fig, ax = plt.subplots(figsize=(12, 6))
air_series.plot(ax=ax, label="Historical", color='black', linewidth=2)

# Add train/test split line
split_time = air_train.end_time()
ax.axvline(split_time, color='red', linestyle='--', linewidth=2,
           label='Train/Test Split', alpha=0.7)

ax.set_title("Air Passengers Dataset: Train/Test Split (80/20)")
ax.set_ylabel("Passengers (thousands)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Generate Forecasts

In [ ]:
# TimesFM 2.5
print("Generating TimesFM 2.5 forecast...")
timesfm_model = TimesFMModel(
    context_length=512,
    device='auto'
)
air_timesfm_forecast = timesfm_model.predict(n=len(air_val), series=air_train, num_samples=NUM_SAMPLES)
air_timesfm_mape = mape(air_val, air_timesfm_forecast)
print(f"TimesFM 2.5 MAPE: {air_timesfm_mape:.2f}%")

In [ ]:
# Chronos 2
print("Generating Chronos 2 forecast...")
chronos_model = ChronosModel(context_length=512)
air_chronos_forecast = chronos_model.predict(n=len(air_val), series=air_train, num_samples=NUM_SAMPLES)
air_chronos_mape = mape(air_val, air_chronos_forecast)
print(f"Chronos 2 MAPE: {air_chronos_mape:.2f}%")

In [ ]:
# Exponential Smoothing (Probabilistic)
print("Generating Exponential Smoothing forecast...")
exp_model = ExponentialSmoothing()
exp_model.fit(air_train)
air_exp_forecast = exp_model.predict(n=len(air_val), num_samples=NUM_SAMPLES)
air_exp_mape = mape(air_val, air_exp_forecast)
print(f"Exponential Smoothing MAPE: {air_exp_mape:.2f}%")

### Compare Forecasts

We compare the models in two ways:
1. **Median Comparison**: Quick comparison of point forecasts across all models
2. **Probabilistic Forecasts**: Individual uncertainty quantification with confidence intervals (50%, 75%, 90%, 95%)

In [ ]:
# Median Comparison - All Models
fig, ax = plt.subplots(figsize=(14, 5))

# Get median forecasts (remove probabilistic samples)
air_timesfm_median = air_timesfm_forecast.quantile_timeseries(quantile=0.5)
air_chronos_median = air_chronos_forecast.quantile_timeseries(quantile=0.5)
air_exp_median = air_exp_forecast.quantile_timeseries(quantile=0.5)

air_val.plot(ax=ax, label="ground truth", color='black', linewidth=2.5)
air_timesfm_median.plot(
    ax=ax,
    label=f"TimesFM 2.5 ({air_timesfm_mape:.2f}% MAPE)",
    color=TIMESFM_COLOR,
    linewidth=2
)
air_chronos_median.plot(
    ax=ax,
    label=f"Chronos 2 ({air_chronos_mape:.2f}% MAPE)",
    color=CHRONOS_COLOR,
    linewidth=2
)
air_exp_median.plot(
    ax=ax,
    label=f"Exp. Smoothing ({air_exp_mape:.2f}% MAPE)",
    color=EXP_COLOR,
    linewidth=2
)
ax.set_title("Air Passengers: Median Forecast Comparison")
ax.set_ylabel("Passengers (thousands)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Probabilistic Forecasts with Confidence Intervals
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# TimesFM 2.5 (native quantile forecasting via quantile head)
air_val.plot(ax=axes[0], label="ground truth", color='black', linewidth=2.5)
air_timesfm_forecast.plot(
    ax=axes[0],
    label="TimesFM 2.5_q0.05-q0.95",
    color=TIMESFM_COLOR
)
axes[0].set_title("TimesFM 2.5: Probabilistic Forecast (Quantile Head)")
axes[0].set_ylabel("Passengers (thousands)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Chronos 2 (sampling-based probabilistic forecasting)
air_val.plot(ax=axes[1], label="ground truth", color='black', linewidth=2.5)
air_chronos_forecast.plot(
    ax=axes[1],
    label="Chronos 2_q0.05-q0.95",
    color=CHRONOS_COLOR
)
axes[1].set_title("Chronos 2: Probabilistic Forecast (Sampling)")
axes[1].set_ylabel("Passengers (thousands)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Exponential Smoothing (probabilistic with uncertainty bands)
air_val.plot(ax=axes[2], label="ground truth", color='black', linewidth=2.5)
air_exp_forecast.plot(
    ax=axes[2],
    label="Exp. Smoothing_q0.05-q0.95",
    color=EXP_COLOR
)
axes[2].set_title("Exponential Smoothing: Probabilistic Forecast")
axes[2].set_ylabel("Passengers (thousands)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Performance Summary

In [ ]:
# Create performance table
air_results = pd.DataFrame({
    'Model': ['TimesFM 2.5', 'Chronos 2', 'Exponential Smoothing'],
    'MAPE (%)': [air_timesfm_mape, air_chronos_mape, air_exp_mape]
})
air_results = air_results.sort_values('MAPE (%)')
print("\nAir Passengers Performance:")
print(air_results.to_string(index=False))

In [ ]:
# Residual Distribution Analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Calculate residuals
timesfm_residuals = (air_val - air_timesfm_median).values().flatten()
chronos_residuals = (air_val - air_chronos_median).values().flatten()
exp_residuals = (air_val - air_exp_median).values().flatten()

# Plot histograms with KDE
axes[0].hist(timesfm_residuals, bins=20, alpha=0.7, color=TIMESFM_COLOR, edgecolor='black')
axes[0].set_title(f'TimesFM 2.5 Residuals\nMean: {timesfm_residuals.mean():.2f}, Std: {timesfm_residuals.std():.2f}')
axes[0].axvline(0, color='black', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

axes[1].hist(chronos_residuals, bins=20, alpha=0.7, color=CHRONOS_COLOR, edgecolor='black')
axes[1].set_title(f'Chronos 2 Residuals\nMean: {chronos_residuals.mean():.2f}, Std: {chronos_residuals.std():.2f}')
axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

axes[2].hist(exp_residuals, bins=20, alpha=0.7, color=EXP_COLOR, edgecolor='black')
axes[2].set_title(f'Exp. Smoothing Residuals\nMean: {exp_residuals.mean():.2f}, Std: {exp_residuals.std():.2f}')
axes[2].axvline(0, color='black', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual (Actual - Predicted)')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Air Passengers: Residual Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Residual Analysis

Understanding forecast errors helps identify model biases and reliability:
- **Centered at zero:** Good - no systematic bias
- **Symmetric distribution:** Good - balanced over/under-prediction
- **Low variance:** Good - consistent accuracy

## 5. Dataset 2: Energy Load (Complex Hourly Patterns)

Energy load data contains complex multi-scale seasonality (daily and weekly patterns) - testing how models handle hierarchical temporal structure.

### Load and Visualize

In [ ]:
# Load dataset (single component, subset for tutorial speed)
energy = EnergyDataset().load()
energy_series = energy['total load actual'][-1000:]  # Use last 1000 points
energy_train, energy_val = energy_series.split_before(SPLIT_RATIO)

print(f"Total length: {len(energy_series)}")
print(f"Training: {len(energy_train)} points")
print(f"Validation: {len(energy_val)} points")

# Visualize with train/test split
fig, ax = plt.subplots(figsize=(12, 6))
energy_series.plot(ax=ax, label="Historical", color='black', linewidth=2)

# Add train/test split line
split_time = energy_train.end_time()
ax.axvline(split_time, color='red', linestyle='--', linewidth=2,
           label='Train/Test Split', alpha=0.7)

ax.set_title("Energy Load Dataset: Train/Test Split (80/20)")
ax.set_ylabel("Load (MW)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Generate Forecasts

In [ ]:
# TimesFM 2.5
print("Generating TimesFM 2.5 forecast...")
energy_timesfm_forecast = timesfm_model.predict(n=len(energy_val), series=energy_train, num_samples=NUM_SAMPLES)
energy_timesfm_mape = mape(energy_val, energy_timesfm_forecast)
print(f"TimesFM 2.5 MAPE: {energy_timesfm_mape:.2f}%")

In [ ]:
# Chronos 2
print("Generating Chronos 2 forecast...")
energy_chronos_forecast = chronos_model.predict(n=len(energy_val), series=energy_train, num_samples=NUM_SAMPLES)
energy_chronos_mape = mape(energy_val, energy_chronos_forecast)
print(f"Chronos 2 MAPE: {energy_chronos_mape:.2f}%")

In [ ]:
# Exponential Smoothing (Probabilistic)
print("Generating Exponential Smoothing forecast...")
exp_model_energy = ExponentialSmoothing()
exp_model_energy.fit(energy_train)
energy_exp_forecast = exp_model_energy.predict(n=len(energy_val), num_samples=NUM_SAMPLES)
energy_exp_mape = mape(energy_val, energy_exp_forecast)
print(f"Exponential Smoothing MAPE: {energy_exp_mape:.2f}%")

### Compare Forecasts

We compare the models in two ways:
1. **Median Comparison**: Quick comparison of point forecasts across all models
2. **Probabilistic Forecasts**: Individual uncertainty quantification with confidence intervals (50%, 75%, 90%, 95%)

In [ ]:
# Median Comparison - All Models
fig, ax = plt.subplots(figsize=(14, 5))

# Get median forecasts (remove probabilistic samples)
energy_timesfm_median = energy_timesfm_forecast.quantile_timeseries(quantile=0.5)
energy_chronos_median = energy_chronos_forecast.quantile_timeseries(quantile=0.5)
energy_exp_median = energy_exp_forecast.quantile_timeseries(quantile=0.5)

energy_val.plot(ax=ax, label="ground truth", color='black', linewidth=2.5)
energy_timesfm_median.plot(
    ax=ax,
    label=f"TimesFM 2.5 ({energy_timesfm_mape:.2f}% MAPE)",
    color=TIMESFM_COLOR,
    linewidth=2
)
energy_chronos_median.plot(
    ax=ax,
    label=f"Chronos 2 ({energy_chronos_mape:.2f}% MAPE)",
    color=CHRONOS_COLOR,
    linewidth=2
)
energy_exp_median.plot(
    ax=ax,
    label=f"Exp. Smoothing ({energy_exp_mape:.2f}% MAPE)",
    color=EXP_COLOR,
    linewidth=2
)
ax.set_title("Energy Load: Median Forecast Comparison")
ax.set_ylabel("Load (MW)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Probabilistic Forecasts with Confidence Intervals
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# TimesFM 2.5 (native quantile forecasting via quantile head)
energy_val.plot(ax=axes[0], label="ground truth", color='black', linewidth=2.5)
energy_timesfm_forecast.plot(
    ax=axes[0],
    label="TimesFM 2.5_q0.05-q0.95",
    color=TIMESFM_COLOR
)
axes[0].set_title("TimesFM 2.5: Probabilistic Forecast (Quantile Head)")
axes[0].set_ylabel("Load (MW)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Chronos 2 (sampling-based probabilistic forecasting)
energy_val.plot(ax=axes[1], label="ground truth", color='black', linewidth=2.5)
energy_chronos_forecast.plot(
    ax=axes[1],
    label="Chronos 2_q0.05-q0.95",
    color=CHRONOS_COLOR
)
axes[1].set_title("Chronos 2: Probabilistic Forecast (Sampling)")
axes[1].set_ylabel("Load (MW)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Exponential Smoothing (probabilistic with uncertainty bands)
energy_val.plot(ax=axes[2], label="ground truth", color='black', linewidth=2.5)
energy_exp_forecast.plot(
    ax=axes[2],
    label="Exp. Smoothing_q0.05-q0.95",
    color=EXP_COLOR
)
axes[2].set_title("Exponential Smoothing: Probabilistic Forecast")
axes[2].set_ylabel("Load (MW)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual Distribution Analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Calculate residuals
timesfm_residuals = (energy_val - energy_timesfm_median).values().flatten()
chronos_residuals = (energy_val - energy_chronos_median).values().flatten()
exp_residuals = (energy_val - energy_exp_median).values().flatten()

# Plot histograms with KDE
axes[0].hist(timesfm_residuals, bins=20, alpha=0.7, color=TIMESFM_COLOR, edgecolor='black')
axes[0].set_title(f'TimesFM 2.5 Residuals\nMean: {timesfm_residuals.mean():.2f}, Std: {timesfm_residuals.std():.2f}')
axes[0].axvline(0, color='black', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

axes[1].hist(chronos_residuals, bins=20, alpha=0.7, color=CHRONOS_COLOR, edgecolor='black')
axes[1].set_title(f'Chronos 2 Residuals\nMean: {chronos_residuals.mean():.2f}, Std: {chronos_residuals.std():.2f}')
axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

axes[2].hist(exp_residuals, bins=20, alpha=0.7, color=EXP_COLOR, edgecolor='black')
axes[2].set_title(f'Exp. Smoothing Residuals\nMean: {exp_residuals.mean():.2f}, Std: {exp_residuals.std():.2f}')
axes[2].axvline(0, color='black', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual (Actual - Predicted)')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Energy Load: Residual Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Residual Analysis

Understanding forecast errors helps identify model biases and reliability:
- **Centered at zero:** Good - no systematic bias
- **Symmetric distribution:** Good - balanced over/under-prediction
- **Low variance:** Good - consistent accuracy

### Performance Summary

In [ ]:
# Create performance table
energy_results = pd.DataFrame({
    'Model': ['TimesFM 2.5', 'Chronos 2', 'Exponential Smoothing'],
    'MAPE (%)': [energy_timesfm_mape, energy_chronos_mape, energy_exp_mape]
})
energy_results = energy_results.sort_values('MAPE (%)')
print("\nEnergy Load Performance:")
print(energy_results.to_string(index=False))

## 6. Dataset 3: Taylor Electricity Demand (Irregular Patterns)

The Taylor dataset contains half-hourly electricity demand with irregular spikes and stochastic behavior - testing how models handle unpredictable patterns.

### Load and Visualize

In [ ]:
# Load dataset (subset for tutorial speed)
from darts.datasets import TaylorDataset
taylor_series = TaylorDataset().load()[-1000:]  # Use last 1000 points
taylor_train, taylor_val = taylor_series.split_before(SPLIT_RATIO)

print(f"Total length: {len(taylor_series)}")
print(f"Training: {len(taylor_train)} points")
print(f"Validation: {len(taylor_val)} points")

# Visualize with train/test split
fig, ax = plt.subplots(figsize=(12, 6))
taylor_series.plot(ax=ax, label="Historical", color='black', linewidth=2)

# Add train/test split line
split_time = taylor_train.end_time()
ax.axvline(split_time, color='red', linestyle='--', linewidth=2,
           label='Train/Test Split', alpha=0.7)

ax.set_title("Taylor Electricity Demand: Train/Test Split (80/20)")
ax.set_ylabel("Demand (MW)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Generate Forecasts

In [ ]:
# TimesFM 2.5
print("Generating TimesFM 2.5 forecast...")
taylor_timesfm_forecast = timesfm_model.predict(n=len(taylor_val), series=taylor_train, num_samples=NUM_SAMPLES)
taylor_timesfm_mape = mape(taylor_val, taylor_timesfm_forecast)
print(f"TimesFM 2.5 MAPE: {taylor_timesfm_mape:.2f}%")

In [ ]:
# Exponential Smoothing (Probabilistic)
print("Generating Exponential Smoothing forecast...")
exp_model_taylor = ExponentialSmoothing()
exp_model_taylor.fit(taylor_train)
taylor_exp_forecast = exp_model_taylor.predict(n=len(taylor_val), num_samples=NUM_SAMPLES)
taylor_exp_mape = mape(taylor_val, taylor_exp_forecast)
print(f"Exponential Smoothing MAPE: {taylor_exp_mape:.2f}%")

In [ ]:
# Chronos 2
print("Generating Chronos 2 forecast...")
taylor_chronos_forecast = chronos_model.predict(n=len(taylor_val), series=taylor_train, num_samples=NUM_SAMPLES)
taylor_chronos_mape = mape(taylor_val, taylor_chronos_forecast)
print(f"Chronos 2 MAPE: {taylor_chronos_mape:.2f}%")

### Compare Forecasts

We compare the models in two ways:
1. **Median Comparison**: Quick comparison of point forecasts across all models
2. **Probabilistic Forecasts**: Individual uncertainty quantification with confidence intervals (50%, 75%, 90%, 95%)

In [ ]:
# Median Comparison - All Models
fig, ax = plt.subplots(figsize=(14, 5))

# Get median forecasts (remove probabilistic samples)
taylor_timesfm_median = taylor_timesfm_forecast.quantile_timeseries(quantile=0.5)
taylor_chronos_median = taylor_chronos_forecast.quantile_timeseries(quantile=0.5)
taylor_exp_median = taylor_exp_forecast.quantile_timeseries(quantile=0.5)

taylor_val.plot(ax=ax, label="ground truth", color='black', linewidth=2.5)
taylor_timesfm_median.plot(
    ax=ax,
    label=f"TimesFM 2.5 ({taylor_timesfm_mape:.2f}% MAPE)",
    color=TIMESFM_COLOR,
    linewidth=2
)
taylor_chronos_median.plot(
    ax=ax,
    label=f"Chronos 2 ({taylor_chronos_mape:.2f}% MAPE)",
    color=CHRONOS_COLOR,
    linewidth=2
)
taylor_exp_median.plot(
    ax=ax,
    label=f"Exp. Smoothing ({taylor_exp_mape:.2f}% MAPE)",
    color=EXP_COLOR,
    linewidth=2
)
ax.set_title("Taylor Electricity Demand: Median Forecast Comparison")
ax.set_ylabel("Demand (MW)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Probabilistic Forecasts with Confidence Intervals
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# TimesFM 2.5 (native quantile forecasting via quantile head)
taylor_val.plot(ax=axes[0], label="ground truth", color='black', linewidth=2.5)
taylor_timesfm_forecast.plot(
    ax=axes[0],
    label="TimesFM 2.5_q0.05-q0.95",
    color=TIMESFM_COLOR
)
axes[0].set_title("TimesFM 2.5: Probabilistic Forecast (Quantile Head)")
axes[0].set_ylabel("Demand (MW)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Chronos 2 (sampling-based probabilistic forecasting)
taylor_val.plot(ax=axes[1], label="ground truth", color='black', linewidth=2.5)
taylor_chronos_forecast.plot(
    ax=axes[1],
    label="Chronos 2_q0.05-q0.95",
    color=CHRONOS_COLOR
)
axes[1].set_title("Chronos 2: Probabilistic Forecast (Sampling)")
axes[1].set_ylabel("Demand (MW)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Exponential Smoothing (probabilistic with uncertainty bands)
taylor_val.plot(ax=axes[2], label="ground truth", color='black', linewidth=2.5)
taylor_exp_forecast.plot(
    ax=axes[2],
    label="Exp. Smoothing_q0.05-q0.95",
    color=EXP_COLOR
)
axes[2].set_title("Exponential Smoothing: Probabilistic Forecast")
axes[2].set_ylabel("Demand (MW)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Residual Analysis

Understanding forecast errors helps identify model biases and reliability:
- **Centered at zero:** Good - no systematic bias
- **Symmetric distribution:** Good - balanced over/under-prediction
- **Low variance:** Good - consistent accuracy

In [ ]:
# Residual Distribution Analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Calculate residuals
timesfm_residuals = (taylor_val - taylor_timesfm_median).values().flatten()
chronos_residuals = (taylor_val - taylor_chronos_median).values().flatten()
exp_residuals = (taylor_val - taylor_exp_median).values().flatten()

# Plot histograms with KDE
axes[0].hist(timesfm_residuals, bins=20, alpha=0.7, color=TIMESFM_COLOR, edgecolor='black')
axes[0].set_title(f'TimesFM 2.5 Residuals\nMean: {timesfm_residuals.mean():.2f}, Std: {timesfm_residuals.std():.2f}')
axes[0].axvline(0, color='black', linestyle='--', linewidth=2)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

axes[1].hist(chronos_residuals, bins=20, alpha=0.7, color=CHRONOS_COLOR, edgecolor='black')
axes[1].set_title(f'Chronos 2 Residuals\nMean: {chronos_residuals.mean():.2f}, Std: {chronos_residuals.std():.2f}')
axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

axes[2].hist(exp_residuals, bins=20, alpha=0.7, color=EXP_COLOR, edgecolor='black')
axes[2].set_title(f'Exp. Smoothing Residuals\nMean: {exp_residuals.mean():.2f}, Std: {exp_residuals.std():.2f}')
axes[2].axvline(0, color='black', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual (Actual - Predicted)')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Taylor Electricity Demand: Residual Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Performance Summary

In [ ]:
# Create performance table
taylor_results = pd.DataFrame({
    'Model': ['TimesFM 2.5', 'Chronos 2', 'Exponential Smoothing'],
    'MAPE (%)': [taylor_timesfm_mape, taylor_chronos_mape, taylor_exp_mape]
})
taylor_results = taylor_results.sort_values('MAPE (%)')
print("\nTaylor Electricity Demand Performance:")
print(taylor_results.to_string(index=False))

## 7. Performance Summary Across All Datasets

Comparing model performance across different data patterns:

In [ ]:
# Create comprehensive comparison table
summary = pd.DataFrame({
    'Dataset': ['Air Passengers', 'Energy Load', 'Taylor Demand'],
    'TimesFM 2.5 MAPE (%)': [air_timesfm_mape, energy_timesfm_mape, taylor_timesfm_mape],
    'Chronos 2 MAPE (%)': [air_chronos_mape, energy_chronos_mape, taylor_chronos_mape],
    'Exp. Smoothing MAPE (%)': [air_exp_mape, energy_exp_mape, taylor_exp_mape]
})

print("\n" + "="*70)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("="*70)
print(summary.to_string(index=False))
print("="*70)

# Calculate average performance
avg_timesfm = summary['TimesFM 2.5 MAPE (%)'].mean()
avg_chronos = summary['Chronos 2 MAPE (%)'].mean()
avg_exp = summary['Exp. Smoothing MAPE (%)'].mean()

print(f"\nAverage MAPE Across All Datasets:")
print(f"  TimesFM 2.5:          {avg_timesfm:.2f}%")
print(f"  Chronos 2:            {avg_chronos:.2f}%")
print(f"  Exp. Smoothing:       {avg_exp:.2f}%")
print("="*70)

### When to Use Each Model

**TimesFM 2.5 excels when:**
- Working with hourly/daily frequency data
- Need fast inference (decoder-only architecture)
- Have limited computational resources
- Series length matches context window well (512-16K points)
- Want native quantile forecasting (10 quantiles via quantile head)

**Chronos 2 excels when:**
- Working with multivariate time series (future support)
- Longer forecast horizons (up to 1024 points)
- Want language model-style attention mechanisms
- Need more granular quantile resolution (21 quantiles via sampling)

**Exponential Smoothing excels when:**
- Need explainability and interpretability
- Working with simple seasonal patterns
- Have domain knowledge to configure parameters
- Computational resources are very limited

**Probabilistic Forecasting:**
- **TimesFM 2.5**: Native quantile forecasting via quantile head (10 quantiles: 0.0-0.9)
- **Chronos 2**: Sampling-based probabilistic forecasting (21 quantiles)
- **Both provide uncertainty quantification** - choice depends on use case and granularity needs

**General Guidance:**
- **Cold start** (new products): Use foundation models
- **Batch forecasting** (1000s of series): Use foundation models
- **Limited data** (<100 points): Foundation models often better
- **Need explainability**: Traditional models
- **Very long history** (10K+ points with local patterns): Consider ensemble of both

## 8. Key Takeaways

**What We Learned:**

1. **Zero-Shot Power**: Foundation models work immediately without training
   - TimesFM 2.5 and Chronos 2 deliver competitive accuracy out-of-the-box
   - No hyperparameter tuning required
   - Ideal for rapid prototyping and cold-start scenarios

2. **Probabilistic Forecasting**: All models provide uncertainty quantification
   - **TimesFM 2.5**: Native quantile forecasting via quantile head (10 quantiles: 0.0-0.9)
   - **Chronos 2**: Sampling-based probabilistic forecasting (21 quantiles)
   - **Traditional models**: Can also generate probabilistic forecasts (num_samples parameter)
   - Different mechanisms but both foundation models capture uncertainty effectively

3. **Dataset Characteristics Matter**: Different patterns favor different approaches
   - Structured seasonality: All models competitive
   - Complex multi-scale patterns: Foundation models show advantage
   - Irregular spikes: Foundation models more robust

4. **Practical Considerations**:
   - Foundation models require more memory (200M+ parameters)
   - Inference time varies: TimesFM 2.5 fastest, Chronos moderate
   - Traditional models remain valuable for explainability
   - Quantile granularity: Chronos 2 (21 quantiles) vs TimesFM 2.5 (10 quantiles)

5. **Simplified API**: TimesFM 2.5 provides a single, production-ready model
   - No version selection needed - one optimized checkpoint
   - Simplified instantiation with sensible defaults
   - Focus on tuning context length rather than model configuration

**Next Steps:**
- Experiment with your own data
- Try different context lengths (multiples of patch_size)
- Explore ensemble approaches combining multiple models
- Read the [Foundation Models User Guide](../docs/userguide/foundation_models.md) for architecture details

## 9. Resources

### Academic Papers
- **TimesFM**: ["A decoder-only foundation model for time-series forecasting"](https://arxiv.org/abs/2310.10688) (Das et al., ICML 2024)
- **Chronos**: ["Chronos: Learning the Language of Time Series"](https://arxiv.org/abs/2403.07815) (Amazon Science, 2024)

### Model Repositories
- [TimesFM GitHub](https://github.com/google-research/timesfm) - Google Research
- [TimesFM HuggingFace](https://huggingface.co/google/timesfm-2.5-200m-pytorch) - Pre-trained model
- [Chronos GitHub](https://github.com/amazon-science/chronos-forecasting) - Amazon Science
- [Chronos HuggingFace](https://huggingface.co/amazon/chronos-t5-base) - Pre-trained model

### Darts Documentation
- [Foundation Models User Guide](../docs/userguide/foundation_models.md) - Concepts and architecture
- [Darts Documentation](https://unit8co.github.io/darts/) - Complete API reference
- [Installation Guide](../INSTALL.md) - Setup instructions

### Related Topics
- [Torch Forecasting Models Guide](https://unit8co.github.io/darts/userguide/torch_forecasting_models.html)
- [Covariates Guide](https://unit8co.github.io/darts/userguide/covariates.html)
- [Probabilistic Forecasting](https://unit8co.github.io/darts/examples/13-TFT-examples.html)